In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Dataset
df = pd.read_csv("/content/credit_risk_dataset.csv")

# Dataset shape
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print()

# Data types
print(df.dtypes)
print()

# Preview
df.head()

Rows: 32581
Columns: 12

person_age                      int64
person_income                   int64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt                       int64
loan_int_rate                 float64
loan_status                     int64
loan_percent_income           float64
cb_person_default_on_file      object
cb_person_cred_hist_length      int64
dtype: object



,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


In [2]:
#Missing values
print("\n Missing values: \n")
missing=df.isnull().sum()
missing_percent=(missing/df.shape[0]*100).round(2)
print(missing_percent)

missing_data=pd.DataFrame({
    "Missing values":missing,
    "Missing %":missing_percent
})
missing_data=missing_data[missing_data["Missing values"]>0]
if missing_data.empty:
  print("No missing values")
else:
  print(missing_data)


 Missing values: 

person_age                    0.00
person_income                 0.00
person_home_ownership         0.00
person_emp_length             2.75
loan_intent                   0.00
loan_grade                    0.00
loan_amnt                     0.00
loan_int_rate                 9.56
loan_status                   0.00
loan_percent_income           0.00
cb_person_default_on_file     0.00
cb_person_cred_hist_length    0.00
dtype: float64
                   Missing values  Missing %
person_emp_length             895       2.75
loan_int_rate                3116       9.56


In [3]:
#Numerical data
duplicates=df.duplicated().sum()
print(f"Number of duplicates: {duplicates}")
print(df.describe().round(2))


Number of duplicates: 165
       person_age  person_income  person_emp_length  loan_amnt  loan_int_rate  \
count    32581.00       32581.00           31686.00   32581.00       29465.00   
mean        27.73       66074.85               4.79    9589.37          11.01   
std          6.35       61983.12               4.14    6322.09           3.24   
min         20.00        4000.00               0.00     500.00           5.42   
25%         23.00       38500.00               2.00    5000.00           7.90   
50%         26.00       55000.00               4.00    8000.00          10.99   
75%         30.00       79200.00               7.00   12200.00          13.47   
max        144.00     6000000.00             123.00   35000.00          23.22   

       loan_status  loan_percent_income  cb_person_cred_hist_length  
count     32581.00             32581.00                    32581.00  
mean          0.22                 0.17                        5.80  
std           0.41                

In [4]:
#Categorical data
categories=["person_home_ownership","loan_intent","loan_grade","cb_person_default_on_file"]
for col in categories:
  print(df[col].value_counts())

person_home_ownership
RENT        16446
MORTGAGE    13444
OWN          2584
OTHER         107
Name: count, dtype: int64
loan_intent
EDUCATION            6453
MEDICAL              6071
VENTURE              5719
PERSONAL             5521
DEBTCONSOLIDATION    5212
HOMEIMPROVEMENT      3605
Name: count, dtype: int64
loan_grade
A    10777
B    10451
C     6458
D     3626
E      964
F      241
G       64
Name: count, dtype: int64
cb_person_default_on_file
N    26836
Y     5745
Name: count, dtype: int64


In [5]:
#Invalid records
invalid_employee=df[df['person_emp_length']>(df['person_age']-16)]
print(f"Number of invalid records: {len(invalid_employee)}")
print(invalid_employee[["person_age", "person_emp_length"]].head(10))

Number of invalid records: 740
     person_age  person_emp_length
0            22              123.0
9            21                6.0
18           23                8.0
65           22                7.0
68           24                9.0
72           21                6.0
125          23                8.0
141          26               11.0
171          21                6.0
210          21              123.0


In [6]:
#Data Cleaning

In [7]:
#remove duplicates
original_rows=len(df)
emp_nulls_before = df['person_emp_length'].isnull().sum()
print(f"emp_length nulls before duplicate removal: {emp_nulls_before}")
print()
df=df.drop_duplicates()
df=df.reset_index(drop=True)
emp_nulls_after = df['person_emp_length'].isnull().sum()
print(f"emp_length nulls after duplicate removal: {emp_nulls_after}")
print(f"nulls lost in duplicate removal: {emp_nulls_before - emp_nulls_after}")
print(f"duplicates removed: {original_rows - len(df)}")


emp_length nulls before duplicate removal: 895

emp_length nulls after duplicate removal: 887
nulls lost in duplicate removal: 8
duplicates removed: 165


In [8]:
#Fixing impossible age values
#Capping at 80 as reasonable maximum for a loan applicant
print(f"Ages above 80:{len(df[df['person_age']>80])}")
df['person_age']=df['person_age'].clip(upper=80)
print(f"Ages above 80:{len(df[df['person_age']>80])}")

Ages above 80:7
Ages above 80:0


In [9]:
#Fixing impossible employement lengths
print(f"Impossible employement lengths: {len(df[df['person_emp_length']>df['person_age']-16])}")
max_possible_emp_length=df['person_age']-16
df['person_emp_length']=df.apply(lambda row:min(row['person_emp_length'],max_possible_emp_length[row.name])
if pd.notnull(row['person_emp_length']) else row['person_emp_length'],axis=1)

print(f"Impossible employement lengths after : {len(df[df['person_emp_length']>(df['person_age']-16)])}")
print(f"Maximum employement length after fix: {df['person_emp_length'].max()}")

Impossible employement lengths: 737
Impossible employement lengths after : 0
Maximum employement length after fix: 41.0


In [10]:
#Handling Employement Null values
df['is_unemployed']=df['person_emp_length'].isnull().astype(int)
df['person_emp_length']=df['person_emp_length'].fillna(0)
print(f"Number of null values in employement length: {df['person_emp_length'].isnull().sum()}")
print(f"Unemployed borrowers flagged as 1: {df['is_unemployed'].sum()}")

Number of null values in employement length: 0
Unemployed borrowers flagged as 1: 887


In [11]:
#Handling Interest Rate nulls
print(f"Missing interet rate values: {df['loan_int_rate'].isnull().sum()}")
df['loan_int_rate']=df.groupby('loan_grade')['loan_int_rate'].transform(lambda x:x.fillna(x.median()))
print(f"Missing interet rate values after: {df['loan_int_rate'].isnull().sum()}")
print("\n Median interest rate by grade:")
print(df.groupby('loan_grade')['loan_int_rate'].median().round(2))


Missing interet rate values: 3095
Missing interet rate values after: 0

 Median interest rate by grade:
loan_grade
A     7.49
B    10.99
C    13.48
D    15.31
E    16.82
F    18.54
G    20.16
Name: loan_int_rate, dtype: float64


In [12]:
#Handling income outliers
Q1=df['person_income'].quantile(0.25)
Q3=df['person_income'].quantile(0.75)
IQR=Q3-Q1
upper_bound=Q3+1.5*IQR
print(f"Income above fence:${upper_bound}")
print(f"Number of outliers:{len(df[df['person_income']>upper_bound])}")
df['person_income']=df['person_income'].clip(upper=upper_bound)
print(f"Max income after capping:${df['person_income'].max()}")

Income above fence:$140232.0
Number of outliers:1478
Max income after capping:$140232


In [13]:
categories

['person_home_ownership',
 'loan_intent',
 'loan_grade',
 'cb_person_default_on_file']

In [14]:
#Standardising categorical columns
for col in categories:
  df[col]=df[col].str.strip().str.upper()
for col in categories:
  print(f"\n{col}:")
  print(df[col].value_counts())


person_home_ownership:
person_home_ownership
RENT        16378
MORTGAGE    13369
OWN          2563
OTHER         106
Name: count, dtype: int64

loan_intent:
loan_intent
EDUCATION            6411
MEDICAL              6042
VENTURE              5682
PERSONAL             5498
DEBTCONSOLIDATION    5189
HOMEIMPROVEMENT      3594
Name: count, dtype: int64

loan_grade:
loan_grade
A    10703
B    10387
C     6438
D     3620
E      963
F      241
G       64
Name: count, dtype: int64

cb_person_default_on_file:
cb_person_default_on_file
N    26686
Y     5730
Name: count, dtype: int64


In [15]:
#Final check

In [16]:
print(f"Row count: {df.shape[0]}")
print(f"Column count: {df.shape[1]}")
print(f"Columns:{list(df.columns)}")
print()

print("Nulls remaining:")
print(df.isnull().sum())
print()

print("Data types:")
print(df.dtypes)
print()

print("stats after cleaning:")
print(df[['person_age', 'person_income',
          'person_emp_length', 'loan_int_rate']].describe().round(2))
print()

print("unique values in categorical columns:")
for col in categories:
    print(f"  {col}: {sorted(df[col].unique())}")
print()

default_rate = df['loan_status'].mean() * 100
print(f"overall default rate: {default_rate:.2f}%")
print(f"total defaulted loans: {df['loan_status'].sum():,}")
print(f"total non-defaulted loans: {(df['loan_status'] == 0).sum():,}")



Row count: 32416
Column count: 13
Columns:['person_age', 'person_income', 'person_home_ownership', 'person_emp_length', 'loan_intent', 'loan_grade', 'loan_amnt', 'loan_int_rate', 'loan_status', 'loan_percent_income', 'cb_person_default_on_file', 'cb_person_cred_hist_length', 'is_unemployed']

Nulls remaining:
person_age                    0
person_income                 0
person_home_ownership         0
person_emp_length             0
loan_intent                   0
loan_grade                    0
loan_amnt                     0
loan_int_rate                 0
loan_status                   0
loan_percent_income           0
cb_person_default_on_file     0
cb_person_cred_hist_length    0
is_unemployed                 0
dtype: int64

Data types:
person_age                      int64
person_income                   int64
person_home_ownership          object
person_emp_length             float64
loan_intent                    object
loan_grade                     object
loan_amnt          

In [17]:
#Saving cleaned dataset
df.to_csv('credit_risk_cleaned.csv', index=False)
print("Cleaned file saved as credit_risk_cleaned.csv")

Cleaned file saved as credit_risk_cleaned.csv


# New Section